In [1]:
import pandas as pd
from pathlib import Path
import requests 

In [24]:
project_root = Path.cwd()
if not Path('data').exists(): 
    project_root = project_root.parent 
print(project_root)

d:\Wu\2026\Project Portfolio\003 Project\grocery-demand-estimation


### Downloading raw ACS variables

In [7]:
# constants
YEAR = '2024'
STATE = '16'
COUNTY = '055'
API_KEY = '71183c4edbddf39da48305b7c7c09f2497c1892b'

In [ ]:
variables = [
    "NAME",         # tract name 
    "B11001_001E",  # household count
    "B25010_001E",  # average household size
    "B19013_001E",  # median household income
    "B01002_001E",  # median age
    "B25003_001E",  # occupied housing units
    "B25003_002E",  # owner occupied
    "B15003_001E",  # population 25+
    "B15003_022E",  # bachelor's
    "B15003_023E",  # master's
    "B15003_024E",  # professional
    "B15003_025E"   # doctorate
]

In [4]:
get_vars = ','.join(variables)
print(get_vars)

NAME,B11001_001E,B25010_001E,B19013_001E,B01002_001E,B25003_001E,B25003_002E,B15003_001E,B15003_022E,B15003_023E,B15003_024E,B15003_025E


#### Build the API URL dynamically

In [14]:
url = (
    f"https://api.census.gov/data/{YEAR}/acs/acs5"
    f"?get={get_vars}"
    "&for=tract:*"
    f"&in=state:{STATE}%20county:{COUNTY}"
    f"&key={API_KEY}"
)
print(url)

https://api.census.gov/data/2024/acs/acs5?get=NAME,B11001_001E,B25010_001E,B19013_001E,B01002_001E,B25003_001E,B25003_002E,B15003_001E,B15003_022E,B15003_023E,B15003_024E,B15003_025E&for=tract:*&in=state:16%20county:055&key=71183c4edbddf39da48305b7c7c09f2497c1892b


#### Send requests to Census API Server

In [15]:
response = requests.get(url)
print(response.status_code)
print(response.text[:300])

200
[["NAME","B11001_001E","B25010_001E","B19013_001E","B01002_001E","B25003_001E","B25003_002E","B15003_001E","B15003_022E","B15003_023E","B15003_024E","B15003_025E","state","county","tract"],
["Census Tract 1.01; Kootenai County; Idaho","1202","2.59","97381","51.3","1202","1058","2342","560","199","41


In [16]:
data = response.json()
type(data)

list

In [18]:
df = pd.DataFrame(data[1:], columns=data[0])
df.head()

,NAME,B11001_001E,B25010_001E,B19013_001E,B01002_001E,B25003_001E,B25003_002E,B15003_001E,B15003_022E,B15003_023E,B15003_024E,B15003_025E,state,county,tract
0,Census Tract 1.01; Kootenai County; Idaho,1202,2.59,97381,51.3,1202,1058,2342,560,199,41,38,16,055,000101
1,Census Tract 1.02; Kootenai County; Idaho,1847,2.56,96493,49.5,1847,1658,3586,619,191,0,26,16,055,000102
2,Census Tract 2.01; Kootenai County; Idaho,1330,2.92,109726,37.8,1330,1065,2776,561,123,14,36,16,055,000201
3,Census Tract 2.02; Kootenai County; Idaho,1312,2.70,92500,53.7,1312,1266,2615,436,275,72,20,16,055,000202
4,Census Tract 2.03; Kootenai County; Idaho,1407,2.41,70843,52.6,1407,1136,2538,620,56,0,0,16,055,000203


#### Create GEOID

In [20]:
df['GEOID'] = (
    df['state'].astype(str).str.zfill(2) +
    df['county'].astype(str).str.zfill(3) + 
    df['tract'].astype(str).str.zfill(6)
)

In [21]:
print(df.shape)
print(df.columns)

(39, 16)
Index(['NAME', 'B11001_001E', 'B25010_001E', 'B19013_001E', 'B01002_001E',
       'B25003_001E', 'B25003_002E', 'B15003_001E', 'B15003_022E',
       'B15003_023E', 'B15003_024E', 'B15003_025E', 'state', 'county', 'tract',
       'GEOID'],
      dtype='str')


#### Save raw ACS data

In [25]:
output_path = project_root / 'data/raw/kootenai_acs2024_raw.csv'

In [27]:
df.to_csv(output_path, index=False)
print(f"saved raw ACS data to {output_path}")

saved raw ACS data to d:\Wu\2026\Project Portfolio\003 Project\grocery-demand-estimation\data\raw\kootenai_acs2024_raw.csv
